# THERMA SEARCH AI — 01 · HIT-UAV veri keşfi

## Amaç ve mevcut durum
Bu öğretici notebook, **model eğitmeden**, kullanıcı tarafından sağlanan HIT-UAV dosyalarını salt okunur biçimde inceler. Varsayılan çalıştırma veri veya ağırlık indirmez. GPU gerekmez.

**Bu teslimde gerçek veri incelenmedi.** Dosya sayıları, çözünürlükler, sınıf dağılımı, bölümler ve koordinat konvansiyonu henüz yerel ölçümle doğrulanmadı. Sentetik doğrulama sonuçları veri seti bulgusu değildir.

## Kaynaklar ve kullanım koşulları
- Esas alınacak sürüm: kullanıcının belirttiği **Zenodo v1.2.1**. [Zenodo HIT-UAV araması](https://zenodo.org/search?q=HIT-UAV); [yayın DOI](https://doi.org/10.1038/s41597-023-02066-6).
- [Resmî GitHub README, sabit commit](https://github.com/suojiashun/HIT-UAV-Infrared-Thermal-Dataset/blob/c5ce70a049d38036fe767b68692c3411ef04e900/README.md): standart `[xc,yc,w,h]` ve döndürülmüş `[xc,yc,w,h,theta]` kutulardan, XML/JSON seçeneklerinden söz eder. Bu beyan, indirilecek arşivin şemasını tek başına kanıtlamaz. GitHub klonu bütün görüntü/etiket dosyalarını sağlamayabilir.
- **Lisans henüz Zenodo üzerinden doğrulanmadı.** Resmî v1.2.1 kaydındaki lisansı, atıf ve yeniden dağıtım koşullarını okuyup saklayın. GitHub kod lisansı ile veri lisansını eşitlemeyin. Belirsizlikte indirmeyi durdurun.
- 2026-11-13: bu geliştirme ortamından Zenodo ve WiSARD proje sitesine erişim ağ politikasıyla engellendi. Colab'daki metadata kontrolü bu boşluğu kapatmak içindir.

WiSARD ilk aşamada yalnızca resmî siteden doğrulanmış **küçük örnek arşiv** ile ele alınmalıdır; 40 GB üzerindeki tam arşiv otomatik indirilmez. Ayrıntılar: `docs/dataset_sources.md`.


## Kurulum
Colab'da proje deposunun URL'sini kendi deponuzdan alın, kod deposunu `/content/therma-search-ai` konumuna klonlayın veya proje klasörünü yükleyin. Bu işlem **HIT-UAV verisini indirmez**. Depo adresi burada tahmin edilmemiştir.

Aşağıdaki hücre otomatik kurulum yapmaz. `PROJECT_ROOT` yolunu ayarlayın. Colab'da gerektiğinde ayrı hücrede `%pip install -r /content/therma-search-ai/requirements.txt` çalıştırın.

Pillow görüntü bütünlüğü ve çözünürlük okumak, Matplotlib dağılım ve kutu çizmek için kullanılır. Dosya, JSON, XML, hash ve test işleri Python standart kütüphanesindedir. nbformat/nbclient/ipykernel notebook doğrulamasını sağlar; PyTorch/Ultralytics bu aşamada gerekli değildir.

In [ ]:
from pathlib import Path
import os
import sys
import json
from collections import Counter
from importlib.metadata import version

# Colab'da /content/therma-search-ai; yerelde kökten veya notebooks/ içinden açılabilir.
candidates = [Path.cwd(), Path.cwd().parent, Path('/content/therma-search-ai')]
PROJECT_ROOT = next((p for p in candidates if (p / 'src/data').is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Proje klasörünü yükleyin ve PROJECT_ROOT yolunu ayarlayın.')
sys.path.insert(0, str(PROJECT_ROOT))
from src.data.inventory import inventory, preview_label
from src.data.download import zenodo_metadata, download_zenodo_file, inspect_zip, extract_zip
from src.data.exploration import run_exploration
from src.data.audit import plot_sample
import matplotlib.pyplot as plt

plt.rcParams.update({'figure.figsize': (9, 4), 'font.size': 11, 'axes.spines.top': False,
                     'axes.spines.right': False})
print({name: version(name) for name in ('Pillow', 'matplotlib', 'nbformat', 'nbclient', 'ipykernel')})

## 1 · İsteğe bağlı metadata doğrulama — arşiv indirmez
Önce resmî Zenodo sayfasında **v1.2.1 sürümüne ait sabit record ID** değerini bulun. Concept DOI veya `latest` yerine bu sürümün kayıt sayfasını kullanın. `CHECK_REMOTE_METADATA=True` ve `RECORD_ID` ayarlayıp yalnızca bu hücreyi çalıştırın.

Kod kayıt kimliğini, HIT-UAV başlığını, sürümü, lisans alanını ve dosya listesini kontrol eder. Lisansın uygunluğunu insan değerlendirir. Ağ/sürüm hatasında başka bir kaynağa geçilmez. Çıktıdaki dosya adlarını ve boyutları okuyun; metadata anlık görüntüsü yerel çıktılara kaydedilir.

In [ ]:
CHECK_REMOTE_METADATA = False
RECORD_ID = None  # Resmî v1.2.1 kayıt sayfasından doğrulayarak girin; burada tahmin edilmez.
WORK_ROOT = PROJECT_ROOT / 'outputs' / 'hit_uav_exploration'
metadata = None
if CHECK_REMOTE_METADATA:
    metadata = zenodo_metadata(str(RECORD_ID), expected_version='1.2.1')
    WORK_ROOT.mkdir(parents=True, exist_ok=True)
    (WORK_ROOT / 'zenodo_metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
    print('Sürüm:', metadata['metadata']['version'], 'Lisans:', metadata['metadata']['license'])
    print('Kaynak DOI:', metadata.get('doi'))
    for entry in metadata['files']:
        print(entry['key'], entry['size'], 'bayt', entry['checksum'])
else:
    print('Metadata isteği kapalı; sürüm ve lisans çevrim içi doğrulanmadı.')

## 2 · Kullanıcının çalıştıracağı ayrı indirme bölümü
**Bu aşamada indirme kapalıdır.** İleride indirme kararı verdiğinizde, metadata ve lisansı inceledikten sonra yalnızca seçtiğiniz dosya için `DOWNLOAD_APPROVED=True` yapın. Dosya adını API çıktısından kopyalayın; boyut limitini açıkça belirleyin. Çalışma alanının ve Drive'ın yeterli boş alana sahip olduğunu kontrol edin.

Dosya parça parça yazılır; yayıncının checksum değeri ve beklenen bayt sayısı eşleşmeden tamamlanmış sayılmaz. Manifest sürüm, kayıt ID, kaynak URL, lisans, yayıncı checksum ve yerel SHA256 içerir. **MD5 kazara bozulmayı kontrol eder; tek başına kaynağın güvenilirliğini kanıtlamaz.**

In [ ]:
DOWNLOAD_APPROVED = False
FILE_KEY = None
MAX_DOWNLOAD_BYTES = 0  # Kullanıcının inceleyerek belirlediği üst sınır.
ARCHIVE_PATH = PROJECT_ROOT / 'data' / 'raw' / 'hit_uav_v1.2.1.zip'
if DOWNLOAD_APPROVED:
    if metadata is None:
        raise RuntimeError('Önce v1.2.1 metadata kontrolünü çalıştırın.')
    if FILE_KEY is None:
        raise ValueError('Metadata çıktısından tam dosya adını seçin.')
    manifest = download_zenodo_file(metadata, FILE_KEY, ARCHIVE_PATH,
                                   approved=True, max_bytes=MAX_DOWNLOAD_BYTES)
    print(manifest)
else:
    print('İndirme yapılmadı.')

### Arşiv envanteri ve ayrı çıkarma onayı
Bu yardımcı yalnızca **ZIP** destekler. Resmî dosya başka türdeyse uzantısını değiştirmeyin; o tür için doğrulanmış bir açıcı ekleyin. Önce listeyi inceleyin. Çıkarma yeni dizine yapılır; mevcut verinin üzerine yazılmaz. Yol güvenliği, açılmış boyut limiti ve ZIP CRC denetlenir. Çok parçalı bir arşiv varsa bu yardımcıyla tek parça gibi açmayın.

In [ ]:
INSPECT_ARCHIVE = False
EXTRACT_APPROVED = False
MAX_UNPACKED_BYTES = 0
EXTRACT_TO = PROJECT_ROOT / 'data' / 'raw' / 'hit_uav_v1.2.1'
if INSPECT_ARCHIVE:
    members = inspect_zip(ARCHIVE_PATH, max_unpacked_bytes=MAX_UNPACKED_BYTES)
    print('Üye sayısı:', len(members), 'Açılmış bayt:', sum(m['bytes'] for m in members))
    print(members[:30])
if EXTRACT_APPROVED:
    if not INSPECT_ARCHIVE:
        raise RuntimeError('Önce arşiv listesini inceleyin.')
    extract_zip(ARCHIVE_PATH, EXTRACT_TO, approved=True, max_unpacked_bytes=MAX_UNPACKED_BYTES)

## 3 · Yerel veri kökü ve gerçek dosya yapısı
Mevcut, bütünlüğü doğrulanmış v1.2.1 verinizi `DATA_ROOT` ile gösterin. Drive bağlamak isterseniz ayrı Colab hücresinde `from google.colab import drive; drive.mount('/content/drive')` kullanın, ardından ilgili yolu girin.

Envanter tüm dosyaları tarar; büyük ağaçta zaman alabilir. Birden fazla kopya veya XML/JSON/OBB gösterimi varsa her biri ayrı kalır. Sonraki aşamada **tek bir standart kutu gösterimi** seçilecektir; aynı görüntünün farklı gösterimleri birleştirilerek sayılmayacaktır.

In [ ]:
DATA_ROOT = Path(os.environ.get('THERMA_DATA_ROOT', str(EXTRACT_TO)))
CONFIG_PATH = Path(os.environ.get('THERMA_CONFIG_PATH', str(PROJECT_ROOT / 'configs/hit_uav.example.json')))
FILE_INVENTORY = None
if DATA_ROOT.is_dir():
    FILE_INVENTORY = inventory(DATA_ROOT)
    print('Uzantılar:', FILE_INVENTORY['extensions'])
    print('Görüntü dosyası adayı:', len(FILE_INVENTORY['images']))
    print('Dizinler (ilk 50):', FILE_INVENTORY['directories'][:50])
    print('Etiket adayları (ilk 30):', FILE_INVENTORY['label_candidates'][:30])
else:
    print('Veri kökü yok. Gerçek veri sayımları yapılmadı:', DATA_ROOT)

### Ham etiket örneğini incele
Listeden bir dosya seçin. JSON anahtarları veya XML alanları şemaya dair kanıt sağlar; yalnızca `.json` uzantısı COCO anlamına gelmez. TXT dosyası da YOLO etiketi veya split listesi olabilir. Birkaç örnek, negatif kare ve farklı split inceleyin.

In [ ]:
LABEL_PREVIEW_RELATIVE_PATH = None  # Envanterden kopyalayın.
if LABEL_PREVIEW_RELATIVE_PATH is not None:
    from src.data.annotations import safe_relative
    preview = preview_label(DATA_ROOT / safe_relative(LABEL_PREVIEW_RELATIVE_PATH))
    print(json.dumps(preview, indent=2, ensure_ascii=False)[:6000])
else:
    print('Etiket örneği seçilmedi; etiket biçimi henüz doğrulanmadı.')

## 4 · Şema ve mevcut split sınırlarını doğrula
`configs/hit_uav.example.json` dosyasını yerel bir çalışma kopyasına alın. İçini **gerçek envanter ve etiket örneklerine göre** doldurun ve `CONFIG_PATH` değerini değiştirin. Özgün train/validation/test adlarını, üyelik dosyalarını ve varsa video/uçuş meta verisini not edin. Yeni split oluşturmayın.

Yapılandırma sözleşmesi (bu bir **API örneğidir; HIT-UAV klasör yapısı iddiası değildir**):
```json
{
  "dataset_version": "1.2.1",
  "schema_confirmed": true,
  "bbox_format": "cxcywh",
  "coordinate_origin": null,
  "splits": {
    "KAYNAKTAKI_SPLIT_ADI": {
      "format": "coco",
      "images": "ENVANTERDEN_GORELI_GORUNTU_DIZINI",
      "annotations": "ENVANTERDEN_GORELI_ETIKET.json"
    }
  }
}
```
- `coco`: JSON içinde `images`, `categories`, `annotations` dizileri ve ID ilişkileri gerekir. Bu şema doğrulanmadıysa seçmeyin. `bbox_format` dosyadaki gerçek anlama göre `xywh`, `cxcywh` veya `xyxy` olmalıdır. Standart COCO `xywh` kullanır; kaynak README'sindeki merkez tanımını da dikkate alarak örneklerle kontrol edin.
- `voc`: `filename`, `size`, `object/name`, `bndbox/xmin,ymin,xmax,ymax` alanlı XML dizini gerekir. `coordinate_origin` için `zero_half_open` veya `one_inclusive` değerini kaynakla doğrulayın. Merkez/açılı XML bu adaptöre uygun değildir.
- Desteklenmeyen şema için keşfi durdurun ve ham örneğe dayalı küçük bir adaptör ekleyin. Beş koordinatlı döndürülmüş kutular sessizce dört koordinata indirgenmez.
- Çıktı tüm adaptörlerde **0 tabanlı sürekli piksel `[x1,y1,x2,y2]`** biçimindedir; kutu sınırı görüntü dışındaysa raporlanır, kırpılmaz. Normalize YOLO biçimine dönüşüm yapılmaz.

`Person`, `Car`, `Bicycle`, `OtherVehicle`, `DontCare` adları aynen sayılır. Gerçek adlar farklıysa `unexpected_classes` altında görünür; eşleme otomatik yapılmaz. DontCare görsel incelemede kesik çizgiyle gösterilir; eğitim/değerlendirme politikası kullanıcı onayını bekler.

In [ ]:
HASH_IMAGES = False  # Tüm okunabilir görüntülerde SHA256: ek disk I/O; isterseniz açın.
CONFIG = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))
REPORT, LOADED = None, {}
if DATA_ROOT.is_dir() and CONFIG.get('schema_confirmed') and CONFIG.get('splits'):
    REPORT, LOADED = run_exploration(DATA_ROOT, CONFIG, hash_images=HASH_IMAGES)
    for split, report in REPORT['splits'].items():
        print(split, {key: report[key] for key in (
            'image_files', 'readable_images', 'annotation_image_records', 'complete', 'count_scope')})
        print('Çözünürlükler:', report['resolutions'], 'Görüntü modları:', report['modes'])
        print('Nesne/sınıf:', report['objects_by_class'])
        print('Görüntü/sınıf:', report['images_by_class'])
        print('Beklenmeyen adlar:', report['unexpected_classes'])
else:
    print('Analiz yapılmadı: veri ve doğrulanmış, dolu split yapılandırması gerekli.')

## 5 · Sınıf dağılımı ve çözünürlükler
**Nesne sayısı**, seçili etiket kaynağındaki kutu sayısıdır; görüntü sayısı veya kişi kimliği sayısı değildir. **Sınıf içeren görüntü sayısı**, o sınıftan en az bir kutusu bulunan etiket kaydı sayısıdır; sınıflar arasında toplanamaz. Bozuk/eksik görüntüye ait ayrıştırılabilir kayıtlar nesne sayımında korunur.

Çözünürlük dağılımı yalnızca başarıyla açılan görüntülerin genişlik × yüksekliğidir. Bunlar algılama başarımı metrikleri değildir; mAP/precision/recall hesaplanmaz. Kısmi etiket kaynağı hatasında dağılım çizilmez.

In [ ]:
if REPORT is not None:
    for split, report in REPORT['splits'].items():
        if report['complete']:
            counts = report['objects_by_class']
            fig, ax = plt.subplots()
            ax.bar(list(counts), list(counts.values()), color='#276b90')
            ax.set(title=f'{split} · Seçili anotasyon kaynağında sınıf dağılımı',
                   xlabel='Orijinal sınıf adı', ylabel='Kutu sayısı')
            ax.tick_params(axis='x', rotation=20)
            fig.tight_layout()
            plt.show()
        else:
            print(split, 'Etiket ayrıştırma hatası: sınıf grafiği atlandı.')
        resolutions = report['resolutions']
        if resolutions:
            fig, ax = plt.subplots()
            ax.bar(list(resolutions), list(resolutions.values()), color='#276b90')
            ax.set(title=f'{split} · Okunabilir görüntü çözünürlükleri',
                   xlabel='Genişlik × yükseklik (piksel)', ylabel='Görüntü sayısı')
            ax.tick_params(axis='x', rotation=45)
            fig.tight_layout()
            plt.show()
else:
    print('Gerçek veri dağılım grafikleri oluşturulmadı.')

## 6 · Kutuları görüntü üstünde kontrol et
Her splitte ilk iki mevcut görüntü deterministik olarak çizilir; bu örnekler tüm etiket kalitesini temsil etmez. Merkez/sol üst yorumunu, görüntü yönünü ve sınırları kontrol edin. Kaynak görüntüyü ve kutuları değiştirmeyin.

Test kümesinde yalnızca bütünlük/şema denetimi yapın. Test örneklerini model, eşik veya hiperparametre seçimi için kullanmayın; gelecekte son değerlendirme için saklayın.

In [ ]:
if REPORT is not None:
    for split, samples in LOADED.items():
        image_root = Path(REPORT['splits'][split]['image_root'])
        shown = 0
        for sample in samples:
            path = image_root / sample.file_name
            if path.is_file():
                try:
                    figure = plot_sample(path, sample)
                    plt.show()
                    shown += 1
                except (OSError, ValueError) as exc:
                    print('Örnek açılamadı:', sample.file_name, str(exc))
            if shown >= 2:
                break
else:
    print('Kutulu örnek gösterimi için doğrulanmış yerel veri bekleniyor.')

## 7 · Kalite ve sızıntı kontrolleri
Aşağıdaki sorunlar adaydır; otomatik düzeltme veya silme yapılmaz.
- `missing_images`: etikette kayıtlı dosya seçili görüntü kökünde yok.
- `unannotated_images`: dosya var, seçili etiket kaynağında kayıt yok; yol/şema seçimi de neden olabilir.
- `empty_annotation_records`: sıfır kutulu kayıt; geçerli negatif olup olmadığı görsel doğrulanmalı.
- `annotation_errors`: parse/ID/şema hatası; sayımlar kısmi olabilir (`complete=False`).
- `issues`: okunamayan görüntü, boyut uyuşmazlığı veya sınır dışı kutu. Kaynak dosyalar düzeltilmez.

SHA256 açıksa aynı bayt içeriğinin split içi/dışı kopyaları raporlanır. Hash kapalıysa **yapılmadı** yazılır. Kopya bulunmaması sızıntı olmadığını kanıtlamaz. Yeniden kodlanmış/yakın kareler, aynı uçuş/video, çekim zamanı, konum ve WiSARD RGB/IR eşlerinin ortak sahneleri ayrıca incelenmelidir. Grup bilgisi varsa özgün split üyeliğiyle çapraz tablo yapılmalı; bilinmiyorsa sonuç 'doğrulanamadı' olarak kalmalıdır. Split değişikliği için kullanıcı onayı gerekir.

In [ ]:
if REPORT is not None:
    for split, report in REPORT['splits'].items():
        for key in ('missing_images', 'unannotated_images', 'empty_annotation_records',
                    'annotation_errors', 'issues'):
            print(split, key, 'adet:', len(report[key]), 'ilk 10:', report[key][:10])
    print('Tam dosya kopyaları:', REPORT['duplicates'])
    print('Açık kontroller:', REPORT['limitations'])
else:
    print('Kalite/sızıntı ölçümü yapılmadı. Uçuş, video, konum ve yakın kare denetimi açık.')

## 8 · Tekrarlanabilir raporu kaydet
Çıktılar kaynak dosyalardan ayrı, Git tarafından yok sayılan `outputs/` altında tutulur. Yapılandırma, sürüm beyanı ve ortam sürümleri rapora eklenir. Çevrim içi metadata ve indirme manifesti yoksa kaynağın bütünlüğü otomatik doğrulanmış sayılmaz. Rapor dosya adları içerebilir; paylaşmadan önce kişisel bilgi kontrolü yapın.

In [ ]:
SAVE_REPORT = False
if SAVE_REPORT and REPORT is not None:
    from datetime import datetime, timezone
    WORK_ROOT.mkdir(parents=True, exist_ok=True)
    REPORT['generated_at_utc'] = datetime.now(timezone.utc).isoformat()
    REPORT['environment'] = {name: version(name) for name in ('Pillow', 'matplotlib', 'nbformat', 'nbclient', 'ipykernel')}
    REPORT['source_verification'] = 'Yerel manifest ve Zenodo kaydı ayrıca karşılaştırılmalı.'
    output_path = WORK_ROOT / 'exploration_report.json'
    output_path.write_text(json.dumps(REPORT, indent=2, ensure_ascii=False), encoding='utf-8')
    print('Kaydedildi:', output_path)
else:
    print('Rapor yazımı kapalı veya analiz henüz yapılmadı.')

## Sonraki adım
1. Colab'da resmî v1.2.1 kaydını ve kullanım koşullarını doğrulayın; sabit kayıt ID ve metadata'yı saklayın.
2. İndirmeye karar verdiğinizde yalnızca onaylı bölümü çalıştırın veya bütünlüğü doğrulanmış mevcut verinizi bağlayın.
3. Ham etiketleri inceleyin; şema, koordinatlar ve özgün split yollarını doğrulayın. Ardından sayımları, kutulu örnekleri ve kalite raporunu birlikte değerlendirin.
4. Uçuş/video/konum sızıntısı ve DontCare politikası için bilimsel kararları belgeleyip kullanıcı onayına sunun.
5. Ancak sonraki aşamada YOLO11s için bir eğitim planı hazırlayın; RF-DETR Small karşılaştırması daha sonra aynı değerlendirme protokolüyle planlanacaktır. Bu notebook eğitim başlatmaz.
